# Notebook 04 : Membership Inference Signal Analysis

This notebook computes the per-subject delta score and checks how well it separates members from non-members before running the full attack in NB05. This is a diagnostic step: if there is no signal here, there is nothing for LiRA to amplify.

The attack signal is motivated by the similarity distribution approach in *"Membership Inference Attacks via Similarity Distribution in Person Re-identification"* (AAAI 2023). For each subject $i$:

$$\delta_i = \overline{P(\text{same})}_{\text{same-person pairs}} - \overline{P(\text{same})}_{\text{diff-person pairs}}$$

The reasoning is that the authentication model was trained on D5 pairs, which include same-person pairs for all attributed member subjects. The LSTM has learned how each training subject authenticates: it returns higher P(same) for their own pairs and lower P(same) when they are compared to others. Non-members were never in training, so this memorisation does not exist for them and their delta sits closer to zero.

This is a subject-level (user-level) attack, sometimes called IIA (Identity Inference Attack) in the gait recognition literature (Milani WIFS 2024). Rather than asking whether a specific window was in training, we ask whether a person was in training, aggregating over all available pairs to get a stable per-subject score.

For reference, the Milani WIFS 2024 paper uses the identification logit l[id_i] from the CNN classification head as the attack signal, reaching AUC ~97.7% (Table I). That approach requires access to CNN internals. Here the attacker only sees P(same) from the authentication interface: a more realistic black-box scenario.

Data used:

| Data | Used for | Subject group |
|---|---|---|
| D5 train pairs (66,542) | Member delta scores | IDs 21-118 |
| D5 test pairs (7,600) | Non-member delta scores | IDs 1-20 |

## How the membership signal is computed

```
For each subject s:

  D5 pairs attributed to s
       │
       ├── same-person pairs  →  [AuthModel]  →  {P(same)_i}  →  μ_same(s)
       │
       └── diff-person pairs  →  [AuthModel]  →  {P(same)_j}  →  μ_diff(s)

  δ(s)  =  μ_same(s)  −  μ_diff(s)

  Members   → CNN built strong representations for them → larger δ  (mean ≈ 0.76)
  Non-members → never in training                       → smaller δ  (mean ≈ 0.50)

  Attack:  δ(s) > threshold  →  predict MEMBER
```

The intuition: the frozen CNN memorises training subjects. Same-person pairs for a member produce consistently high P(same); for a non-member the model has no memorised signal and the scores are weaker. The difference of means aggregates this into a single stable score per subject.

In [ ]:
import sys
sys.path.insert(0, '..')

import json, logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch

from src.data.dataset      import load_dataset
from src.data.auth_dataset import load_auth_dataset, normalize_auth
from src.models.gait_cnn   import GaitCNN
from src.models.auth_model import AuthModel

# ── paths ──
DATA_ROOT  = Path('../data')
LOG_DIR    = Path('../logs')
CKPT_DIR   = Path('../checkpoints')
RESULT_DIR = Path('../results')
for d in [LOG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── logging ──
log = logging.getLogger('nb04')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR / '04_mia_signal.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh)
log.addHandler(sh)

BATCH_SIZE = 512
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(LOG_DIR / 'subject_split.json') as f:
    split_info = json.load(f)
member_ids    = split_info['train_ids']     # IDs 21–118
nonmember_ids = split_info['held_out_ids']  # IDs 1–20

log.info('=== Notebook 04 — MIA Signal via D5 Authentication Pairs ===')
log.info(f'Device: {DEVICE}')
log.info(f'Members:     {len(member_ids)} subjects  IDs {min(member_ids)}..{max(member_ids)}')
log.info(f'Non-members: {len(nonmember_ids)} subjects  IDs {min(nonmember_ids)}..{max(nonmember_ids)}')

## 1. Load the Model and D5 Pairs

Load the trained authentication model from NB03 (frozen CNN + LSTM) and the full D5 pair dataset. Normalisation statistics from NB03 are reused here.

In [ ]:
# ── Load auth model ──
norm_data  = np.load(LOG_DIR / 'auth_norm_stats.npz')
norm_mean  = norm_data['mean']   # (1, 6, 1)
norm_std   = norm_data['std']    # (1, 6, 1)

cnn   = GaitCNN(n_classes=98)
model = AuthModel(cnn_encoder=cnn).to(DEVICE)
model.load_state_dict(torch.load(CKPT_DIR / 'auth_model.pt', map_location='cpu'))
model.eval()
log.info('Auth model loaded — eval mode, no weight updates')

# ── Load D5 pairs ──
D5_ROOT = str(DATA_ROOT / 'Dataset #5')
X1_tr, X2_tr, y_tr = load_auth_dataset(D5_ROOT, 'train')   # 66,542 member pairs
X1_te, X2_te, y_te = load_auth_dataset(D5_ROOT, 'test')    # 7,600 non-member pairs

# Normalize with D5 train stats (from NB03 — never recomputed)
X1_tr_n, X2_tr_n, _ = normalize_auth(X1_tr, X2_tr, (norm_mean, norm_std))
X1_te_n, X2_te_n, _ = normalize_auth(X1_te, X2_te, (norm_mean, norm_std))

log.info(f'D5 train: {len(y_tr):,} pairs  same={(y_tr==1).sum():,}  diff={(y_tr==0).sum():,}')
log.info(f'D5 test:  {len(y_te):,} pairs  same={(y_te==1).sum():,}  diff={(y_te==0).sum():,}')

print(f'D5 train: {len(y_tr):,} pairs  ({(y_tr==1).sum():,} same / {(y_tr==0).sum():,} diff)')
print(f'D5 test:  {len(y_te):,} pairs  ({(y_te==1).sum():,} same / {(y_te==0).sum():,} diff)')

## 2. D5 Pair Correlation Structure

The D5 pair structure has a non-obvious property that affects the delta score. Same-person pairs in D5 are anti-correlated (different gait cycles, correlation around -0.08) while different-person pairs are highly correlated (same gait phase, correlation around +0.86). The LSTM learned this inverted structure and assigns high P(same) to the anti-correlated pairs. This means the delta score is positive for everyone: but members have a systematically larger delta because the frozen CNN built stronger representations for them during Phase 1.

In [ ]:
def pair_correlations(X1, X2, y, n=500):
    same_idx = np.where(y == 1)[0][:n]
    diff_idx = np.where(y == 0)[0][:n]
    corr_same = np.array([np.corrcoef(X1[i].flatten(), X2[i].flatten())[0, 1] for i in same_idx])
    corr_diff = np.array([np.corrcoef(X1[i].flatten(), X2[i].flatten())[0, 1] for i in diff_idx])
    return corr_same, corr_diff

corr_tr_same, corr_tr_diff = pair_correlations(X1_tr, X2_tr, y_tr)
corr_te_same, corr_te_diff = pair_correlations(X1_te, X2_te, y_te)

log.info(f'D5 train — same-pair corr: mean={corr_tr_same.mean():.3f}  diff-pair corr: mean={corr_tr_diff.mean():.3f}')
log.info(f'D5 test  — same-pair corr: mean={corr_te_same.mean():.3f}  diff-pair corr: mean={corr_te_diff.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bins = np.linspace(-1, 1, 40)

for ax, cs, cd, title in [
    (axes[0], corr_tr_same, corr_tr_diff,
     f'D5 Train (member pairs)\nsame corr={corr_tr_same.mean():.3f}  diff corr={corr_tr_diff.mean():.3f}'),
    (axes[1], corr_te_same, corr_te_diff,
     f'D5 Test (non-member pairs, cross-session)\nsame corr={corr_te_same.mean():.3f}  diff corr={corr_te_diff.mean():.3f}'),
]:
    ax.hist(cs, bins=bins, alpha=0.7, color='#2ecc71', label=f'Same-person  mean={cs.mean():.3f}', density=True)
    ax.hist(cd, bins=bins, alpha=0.7, color='#e74c3c', label=f'Diff-person  mean={cd.mean():.3f}', density=True)
    ax.axvline(cs.mean(), color='#27ae60', linestyle='--', linewidth=1.5)
    ax.axvline(cd.mean(), color='#c0392b', linestyle='--', linewidth=1.5)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Intra-pair Pearson correlation')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('D5 Pair Construction: Same pairs are ANTI-correlated, Diff pairs are HIGHLY correlated\n'
             '(LSTM learned: anti-correlated pair → same person; highly correlated pair → different person)',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(RESULT_DIR / '04_d5_pair_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
log.info('Figure saved: results/04_d5_pair_correlation.png')
print(f'D5 train: same corr={corr_tr_same.mean():.3f}  diff corr={corr_tr_diff.mean():.3f}')
print(f'D5 test:  same corr={corr_te_same.mean():.3f}  diff corr={corr_te_diff.mean():.3f}')
print('→ The LSTM learned that anti-correlated windows = same person (and vice versa).')

## 3. Score All D5 Pairs

Run the full D5 train and test sets through the model and collect P(same) for each pair. Forward pass only, no weight updates.

In [ ]:
def score_pairs(X1, X2):
    """Score all pairs → P(same person). Model is frozen in eval()."""
    out = []
    with torch.no_grad():
        for s in range(0, len(X1), BATCH_SIZE):
            e = min(s + BATCH_SIZE, len(X1))
            out.append(model.similarity(
                torch.from_numpy(X1[s:e]).float().to(DEVICE),
                torch.from_numpy(X2[s:e]).float().to(DEVICE),
            ).cpu().numpy())
    return np.concatenate(out)

print('Scoring D5 train (member pairs)...')
s_tr = score_pairs(X1_tr_n, X2_tr_n)
print('Scoring D5 test  (non-member pairs)...')
s_te = score_pairs(X1_te_n, X2_te_n)

tr_same_mean = s_tr[y_tr == 1].mean()
tr_diff_mean = s_tr[y_tr == 0].mean()
te_same_mean = s_te[y_te == 1].mean()
te_diff_mean = s_te[y_te == 0].mean()

log.info(f'D5 train — same mean={tr_same_mean:.3f}  diff mean={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}')
log.info(f'D5 test  — same mean={te_same_mean:.3f}  diff mean={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}')

print(f'\nD5 train (members):      same={tr_same_mean:.3f}  diff={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}')
print(f'D5 test  (non-members):  same={te_same_mean:.3f}  diff={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}')
print(f'Reference NB03 D5 test AUC: 0.9033')

## 4. Attribute Pairs to Subjects

D5 pairs do not directly label which subject each window belongs to, so attribution is recovered via a fingerprint lookup: the first 32 values of each window rounded to 1 decimal place. D5 train windows trace back to Dataset #1 (member data, same session) and D5 test windows trace back to Dataset #2 (non-member data, different session).

In [ ]:
def window_fingerprint(X):
    """32-value fingerprint rounded to 1 decimal — robust to float32 loading noise (≈5e-6)."""
    return [tuple(np.round(x.flatten()[:32], 1)) for x in X]

def build_lookup(X, y):
    return {k: int(v) for k, v in zip(window_fingerprint(X), y)}

# ── Load source datasets for fingerprint lookup ──
print('Building lookup: D1 (member source)...')
X1_d1, y_d1   = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'train')
X1_d1t, y_d1t = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'test')
D1_all  = np.concatenate([X1_d1,  X1_d1t])
y_d1_all = np.concatenate([y_d1, y_d1t])
lut_d1  = build_lookup(D1_all, y_d1_all)

print('Building lookup: D2 (non-member source)...')
X2_d2, y_d2   = load_dataset(str(DATA_ROOT / 'Dataset #2'), 'train')
X2_d2t, y_d2t = load_dataset(str(DATA_ROOT / 'Dataset #2'), 'test')
D2_all  = np.concatenate([X2_d2,  X2_d2t])
y_d2_all = np.concatenate([y_d2, y_d2t])
lut_d2  = build_lookup(D2_all, y_d2_all)
lut_d1_nm = {k: v for k, v in lut_d1.items() if v in set(nonmember_ids)}

log.info(f'D1 lookup: {len(lut_d1):,} entries  D2 lookup: {len(lut_d2):,} entries')

# ── Attribute D5 pairs ──
def lookup_arr(keys, lut):
    return np.array([lut.get(k, -1) for k in keys])

fp1_tr, fp2_tr = window_fingerprint(X1_tr), window_fingerprint(X2_tr)
fp1_te, fp2_te = window_fingerprint(X1_te), window_fingerprint(X2_te)

a1_tr = lookup_arr(fp1_tr, lut_d1)
a2_tr = lookup_arr(fp2_tr, lut_d1)

a1_te = lookup_arr(fp1_te, lut_d2)
a2_te = lookup_arr(fp2_te, lut_d2)
# D1-nm fallback for D5 test (some non-member windows also in D1)
a1_te_fb = lookup_arr(fp1_te, lut_d1_nm)
a2_te_fb = lookup_arr(fp2_te, lut_d1_nm)
a1_te[a1_te == -1] = a1_te_fb[a1_te == -1]
a2_te[a2_te == -1] = a2_te_fb[a2_te == -1]

match_tr = (a1_tr != -1).mean()
match_te = (a1_te != -1).mean()
log.info(f'D5 train attribution (X1 → D1): {match_tr:.1%}')
log.info(f'D5 test  attribution (X1 → D2): {match_te:.1%}')
print(f'D5 train attribution (X1 → D1): {match_tr:.1%}  (100% = all member windows found in D1)')
print(f'D5 test  attribution (X1 → D2): {match_te:.1%}  (~34% from D4 which is not in our data)')

## 5. Compute Per-Subject Delta

For each attributed subject, group pairs into same-person and different-person, compute the mean P(same) for each group, and take the difference. This gives one delta score per subject.

In [ ]:
def compute_deltas(a1, a2, scores, y, subject_set):
    """Per-subject delta = mean P(same-pair) - mean P(diff-pair).

    Same pairs: attributed via X1; X2 used as fallback if X1 is unmatched.
    Diff pairs: attributed to X1's subject (primary viewpoint of the pair).
    """
    pos_scores = defaultdict(list)
    neg_scores = defaultdict(list)
    for i in range(len(y)):
        s1, s2 = int(a1[i]), int(a2[i])
        if y[i] == 1:  # same-person pair
            sid = s1 if s1 in subject_set else (s2 if s2 in subject_set else -1)
            if sid != -1:
                pos_scores[sid].append(scores[i])
        else:  # diff-person pair
            if s1 in subject_set:
                neg_scores[s1].append(scores[i])
    
    deltas = {}
    for sid in subject_set:
        pos = np.array(pos_scores[sid])
        neg = np.array(neg_scores[sid])
        if len(pos) > 0 and len(neg) > 0:
            deltas[sid] = {
                'delta':    float(pos.mean() - neg.mean()),
                'pos_mean': float(pos.mean()),
                'neg_mean': float(neg.mean()),
                'n_pos':    len(pos),
                'n_neg':    len(neg),
            }
    return deltas

m_deltas  = compute_deltas(a1_tr, a2_tr, s_tr, y_tr, set(member_ids))
nm_deltas = compute_deltas(a1_te, a2_te, s_te, y_te, set(nonmember_ids))

n_m_attr  = len(m_deltas)
n_nm_attr = len(nm_deltas)

m_arr  = np.array([v['delta'] for v in m_deltas.values()])
nm_arr = np.array([v['delta'] for v in nm_deltas.values()])

log.info(f'Members attributed:     {n_m_attr}/98  (14 subjects absent from D5 train — Phase 1 only)')
log.info(f'Non-members attributed: {n_nm_attr}/20')
log.info(f'Member    delta: mean={m_arr.mean():.4f}  std={m_arr.std():.4f}  >0: {(m_arr>0).sum()}/{len(m_arr)}')
log.info(f'Non-member delta: mean={nm_arr.mean():.4f}  std={nm_arr.std():.4f}  >0: {(nm_arr>0).sum()}/{len(nm_arr)}')

print(f'\nMembers attributed:     {n_m_attr}/98')
print(f'Non-members attributed: {n_nm_attr}/20')
print(f'\nMember    delta: mean={m_arr.mean():.4f}  std={m_arr.std():.4f}  >0: {(m_arr>0).sum()}/{len(m_arr)}')
print(f'Non-member delta: mean={nm_arr.mean():.4f}  std={nm_arr.std():.4f}  >0: {(nm_arr>0).sum()}/{len(nm_arr)}')
print(f'Delta gap (member - non-member): {m_arr.mean()-nm_arr.mean():.4f}')

# Note on missing members
missing_m = [s for s in member_ids if s not in m_deltas]
if missing_m:
    log.info(f'Subjects not in D5 train (Phase 1 only, no authentication pairs): {missing_m}')
    print(f'\nNote: {len(missing_m)} subjects not attributed — their D1 windows are absent from D5 train pairs.')

## 6. Visualise the Delta Distributions

Two plots: a histogram comparing member and non-member delta distributions, and a ranked bar chart of individual delta scores. The histogram shows the separation between groups; the bar chart reveals which specific subjects are difficult to classify.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: delta distribution
ax = axes[0]
bins = np.linspace(0, 1, 30)
ax.hist(nm_arr, bins=bins, alpha=0.7, color='#e74c3c',
        label=f'Non-members (n={len(nm_arr)})  mean={nm_arr.mean():.3f}', density=True)
ax.hist(m_arr,  bins=bins, alpha=0.7, color='#3498db',
        label=f'Members (n={len(m_arr)})  mean={m_arr.mean():.3f}', density=True)
ax.axvline(m_arr.mean(),  color='#3498db', linestyle='--', linewidth=1.5)
ax.axvline(nm_arr.mean(), color='#e74c3c', linestyle='--', linewidth=1.5)
ax.set_xlabel('Delta = mean P(same-pair) − mean P(diff-pair)')
ax.set_ylabel('Density')
ax.set_title(f'Delta Score Distribution\n'
             f'Members (D5 train): {m_arr.mean():.3f}  |  Non-members (D5 test): {nm_arr.mean():.3f}\n'
             f'Gap = {m_arr.mean()-nm_arr.mean():.3f}  ← MIA signal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

# Right: aggregate score bars (same vs diff per group)
ax = axes[1]
m_pos_agg  = np.mean([v['pos_mean'] for v in m_deltas.values()])
m_neg_agg  = np.mean([v['neg_mean'] for v in m_deltas.values()])
nm_pos_agg = np.mean([v['pos_mean'] for v in nm_deltas.values()])
nm_neg_agg = np.mean([v['neg_mean'] for v in nm_deltas.values()])

x = np.array([0, 1]); w = 0.3
ax.bar(x - w/2, [m_pos_agg,  nm_pos_agg], w, color=['#3498db', '#e74c3c'], alpha=0.9,
       label='Same-pair mean P(same)')
ax.bar(x + w/2, [m_neg_agg,  nm_neg_agg], w, color=['#3498db', '#e74c3c'], alpha=0.4,
       hatch='//', label='Diff-pair mean P(same)')
for xi, pos, neg in zip(x, [m_pos_agg, nm_pos_agg], [m_neg_agg, nm_neg_agg]):
    ax.annotate(f'{pos:.3f}', (xi - w/2, pos + 0.02), ha='center', fontsize=9)
    ax.annotate(f'{neg:.3f}', (xi + w/2, neg + 0.02), ha='center', fontsize=9)
    ax.annotate(f'Δ={pos-neg:.3f}', (xi, max(pos, neg) + 0.08), ha='center', fontsize=10, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Members\n(D5 train, IDs 21–118)', 'Non-members\n(D5 test, IDs 1–20)'])
ax.set_ylabel('Mean P(same person)')
ax.set_title('Same-pair vs Diff-pair Scores by Group\n'
             'Both groups positive delta — members delta much higher')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULT_DIR / '04_delta_distribution.png', dpi=150)
plt.show()
log.info(f'Figure saved: results/04_delta_distribution.png')
log.info(f'Delta gap: {m_arr.mean()-nm_arr.mean():.4f}')

all_ids    = list(m_deltas.keys())   + list(nm_deltas.keys())
all_deltas = list(m_arr)             + list(nm_arr)
all_labels = ['member']*len(m_arr)   + ['non-member']*len(nm_arr)

order = np.argsort(all_deltas)[::-1]   # highest delta (members) on left
sorted_d = np.array(all_deltas)[order]
sorted_l = [all_labels[i] for i in order]
colors   = ['#3498db' if l == 'member' else '#e74c3c' for l in sorted_l]

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(sorted_d)), sorted_d, color=colors, width=0.8)
ax.axhline(m_arr.mean(),  color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Member mean ({m_arr.mean():.3f})')
ax.axhline(nm_arr.mean(), color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Non-member mean ({nm_arr.mean():.3f})')
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.set_xlabel(f'All {len(sorted_d)} attributed subjects (ranked by delta, highest left)')
ax.set_ylabel('Delta = mean P(same-pair) − mean P(diff-pair)')
ax.set_title(f'Per-Subject Delta Scores: Blue=member (D5 train)  Red=non-member (D5 test)\n'
             f'Delta gap = {m_arr.mean()-nm_arr.mean():.3f}  |  '
             f'Members>0: {(m_arr>0).sum()}/{len(m_arr)}  '
             f'Non-members>0: {(nm_arr>0).sum()}/{len(nm_arr)}')
ax.set_xticks([])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULT_DIR / '04_per_subject_delta.png', dpi=150)
plt.show()
log.info('Figure saved: results/04_per_subject_delta.png')

## 7. Attack AUC : Delta as a Classifier

Use the delta score directly as an attack score: higher delta means predict member. The ROC curve and AUC measure how well this simple threshold separates the two groups.

attack_scores = np.concatenate([m_arr,  nm_arr])
attack_labels = np.concatenate([np.ones(len(m_arr)), np.zeros(len(nm_arr))])

fpr, tpr, thresholds = roc_curve(attack_labels, attack_scores)
mia_auc = sk_auc(fpr, tpr)

# Youden's J → optimal threshold
j = tpr - fpr
opt_idx = np.argmax(j)
opt_thr  = thresholds[opt_idx]
opt_tpr  = tpr[opt_idx]
opt_fpr  = fpr[opt_idx]

# TPR @ fixed FPR points
tpr_at_10 = float(tpr[np.searchsorted(fpr, 0.10)])
tpr_at_20 = float(tpr[np.searchsorted(fpr, 0.20)])

log.info(f'MIA AUC (delta score, D5 pairs): {mia_auc:.4f}')
log.info(f'Optimal threshold (Youden J): {opt_thr:.4f}  TPR={opt_tpr:.3f}  FPR={opt_fpr:.3f}')
log.info(f'TPR @ FPR=0.10: {tpr_at_10:.3f}   TPR @ FPR=0.20: {tpr_at_20:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: ROC curve
ax = axes[0]
ax.plot(fpr, tpr, color='#8e44ad', linewidth=2,
        label=f'MIA ROC (D5 delta)  AUC = {mia_auc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random (0.500)')
ax.scatter([opt_fpr], [opt_tpr], color='orange', s=80, zorder=5,
           label=f'Optimal thr = {opt_thr:.3f}\n(TPR={opt_tpr:.3f}, FPR={opt_fpr:.3f})')
ax.axvline(0.10, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(0.20, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xlabel('False Positive Rate (non-members called members)')
ax.set_ylabel('True Positive Rate (members correctly identified)')
ax.set_title(f'MIA ROC Curve: Delta Score on D5 Pairs\n'
             f'AUC = {mia_auc:.4f}  (baseline: 0.5000)\n'
             f'TPR@FPR=0.10: {tpr_at_10:.3f}  |  TPR@FPR=0.20: {tpr_at_20:.3f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: score distributions with optimal threshold
ax = axes[1]
bins = np.linspace(0, 1, 25)
ax.hist(nm_arr, bins=bins, alpha=0.7, color='#e74c3c', label=f'Non-members (n={len(nm_arr)})', density=True)
ax.hist(m_arr,  bins=bins, alpha=0.7, color='#3498db', label=f'Members (n={len(m_arr)})',       density=True)
ax.axvline(opt_thr, color='orange', linestyle='-', linewidth=2, label=f'Threshold = {opt_thr:.3f}')
ax.set_xlabel('Delta score')
ax.set_ylabel('Density')
ax.set_title('Score Distributions: Members vs Non-members\n'
             'Members (blue) cluster higher → correctly identified by delta threshold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '04_mia_roc.png', dpi=150)
plt.show()
log.info('Figure saved: results/04_mia_roc.png')

print(f'\nMIA AUC: {mia_auc:.4f}  (random baseline: 0.5000)')
print(f'Optimal threshold: {opt_thr:.4f}')
print(f'TPR @ FPR=0.10: {tpr_at_10:.3f}')
print(f'TPR @ FPR=0.20: {tpr_at_20:.3f}')

## 8. Save Scores for NB05

Save per-subject delta scores and the AUC to `logs/04_mia_scores.npz`. NB05 loads this file as the input for the LiRA attack.

m_ids_arr  = np.array(sorted(m_deltas.keys()),  dtype=np.int32)
nm_ids_arr = np.array(sorted(nm_deltas.keys()), dtype=np.int32)
m_scores   = np.array([m_deltas[s]['delta']  for s in m_ids_arr],  dtype=np.float32)
nm_scores  = np.array([nm_deltas[s]['delta'] for s in nm_ids_arr], dtype=np.float32)

np.savez(
    LOG_DIR / '04_mia_scores.npz',
    member_ids       = m_ids_arr,
    member_scores    = m_scores,
    nonmember_ids    = nm_ids_arr,
    nonmember_scores = nm_scores,
    mia_auc          = np.float32(mia_auc),
    delta_gap        = np.float32(m_arr.mean() - nm_arr.mean()),
)
log.info(f'Scores saved: logs/04_mia_scores.npz')

delta_gap = float(m_arr.mean() - nm_arr.mean())

summary = f"""
=== NOTEBOOK 04 SUMMARY ===

D5-BASED MIA SIGNAL (correct evaluation on D5-style pairs)
  Members:     {len(m_arr)} attributed subjects (IDs 21-118: 14/98 absent from D5 train)
  Non-members: {len(nm_arr)} subjects (IDs 1-20)

D5 pair structure:
  Same-pair correlation: {corr_tr_same.mean():.3f} (anti-correlated: different gait phases)
  Diff-pair correlation: {corr_tr_diff.mean():.3f} (highly correlated: same gait phase, different person)

Score distributions:
  D5 train (members):     same={tr_same_mean:.3f}  diff={tr_diff_mean:.3f}  delta={tr_same_mean-tr_diff_mean:.3f}
  D5 test  (non-members): same={te_same_mean:.3f}  diff={te_diff_mean:.3f}  delta={te_same_mean-te_diff_mean:.3f}

Per-subject delta:
  Members:     mean={m_arr.mean():.4f}  std={m_arr.std():.4f}  >0: {(m_arr>0).sum()}/{len(m_arr)}
  Non-members: mean={nm_arr.mean():.4f}  std={nm_arr.std():.4f}  >0: {(nm_arr>0).sum()}/{len(nm_arr)}
  Delta gap (member - non-member): {delta_gap:.4f}

MIA ATTACK RESULTS:
  AUC = {mia_auc:.4f}  (random: 0.5000)
  Optimal threshold: {opt_thr:.4f}  TPR={opt_tpr:.3f}  FPR={opt_fpr:.3f}
  TPR @ FPR=0.10: {tpr_at_10:.3f}   TPR @ FPR=0.20: {tpr_at_20:.3f}

Scores saved: logs/04_mia_scores.npz
NB05 uses delta directly as attack score (higher delta → predict member).
"""
print(summary)
log.info(summary)

## 9. LaTeX Macros

Write key metrics to `latex/generated/nb04_metrics.tex` for direct use in the thesis.

In [ ]:
from src.utils.latex_writer import write_latex_metrics

write_latex_metrics('nb04', {
    # Attribution
    'miaMemberAttributed':          n_m_attr,
    'miaMemberTotal':               len(member_ids),
    'miaNonMemberAttributed':       n_nm_attr,
    # D5 pair correlation
    'miaD5SamePairCorr':            f'{corr_tr_same.mean():.3f}',
    'miaD5DiffPairCorr':            f'{corr_tr_diff.mean():.3f}',
    # Aggregate scores
    'miaMemberSameMean':            f'{tr_same_mean:.3f}',
    'miaMemberDiffMean':            f'{tr_diff_mean:.3f}',
    'miaMemberAggregateDelta':      f'{tr_same_mean-tr_diff_mean:.3f}',
    'miaNonMemberSameMean':         f'{te_same_mean:.3f}',
    'miaNonMemberDiffMean':         f'{te_diff_mean:.3f}',
    'miaNonMemberAggregateDelta':   f'{te_same_mean-te_diff_mean:.3f}',
    # Per-subject delta
    'miaMemberDeltaMean':           f'{m_arr.mean():.4f}',
    'miaMemberDeltaStd':            f'{m_arr.std():.4f}',
    'miaNonMemberDeltaMean':        f'{nm_arr.mean():.4f}',
    'miaNonMemberDeltaStd':         f'{nm_arr.std():.4f}',
    'miaDeltaGap':                  f'{delta_gap:.4f}',
    # MIA AUC
    'miaAUC':                       f'{mia_auc:.4f}',
    'miaOptimalThreshold':          f'{opt_thr:.4f}',
    'miaOptimalTPR':                f'{opt_tpr:.3f}',
    'miaOptimalFPR':                f'{opt_fpr:.3f}',
    'miaTPRatFPR10':                f'{tpr_at_10:.3f}',
    'miaTPRatFPR20':                f'{tpr_at_20:.3f}',
}, output_dir='../latex/generated', log=log)
print('LaTeX macros written.')

## 10. Session-Controlled Validation

The main evaluation uses D5 train pairs for members (windows from D1, same recording session) and D5 test pairs for non-members (windows from D2, a different session). This creates a potential confound: non-member delta scores might be lower because cross-session authentication is genuinely harder, not because they were absent from training.

To check this Authentication pairs are built for non-members sourced entirely from D1: the same dataset and session as members: by selecting window pairs with low intra-pair correlation to match D5's structure. If the membership gap survives this session-matched comparison, the signal is genuine memorisation rather than a session artefact.

Six non-member subjects (IDs 5, 6, 9, 15, 16, 20) have very consistent gait in D1 and cannot produce D5-style anti-correlated pairs. The controlled evaluation covers the remaining 14 subjects.

In [ ]:
# ── Session-controlled evaluation: D1-curated pairs for non-members ──
# Same-person pairs: low correlation  (match D5 train same: mean≈0.234)
# Diff-person pairs: high correlation  (match D5 train diff: mean≈0.762)
SAME_CORR_THRESH = 0.50   # same pairs: corr < this
DIFF_CORR_THRESH = 0.70   # diff pairs: corr > this
N_SAME_TARGET    = 200
N_DIFF_TARGET    = 200
MIN_PAIRS        = 10

# D1_all and y_d1_all already loaded in cell 9; shape (N, 6, 128)
nm_mask_d1 = np.isin(y_d1_all, nonmember_ids)
X_d1_nm    = D1_all[nm_mask_d1]
y_d1_nm    = y_d1_all[nm_mask_d1]

def sample_filtered_pairs(wins1, wins2, n_target, corr_lo=None, corr_hi=None, rng=None, n_attempts=15000):
    """Sample pairs with correlation in (corr_hi, corr_lo) — exclusive."""
    n1, n2 = len(wins1), len(wins2)
    pairs, seen = [], set()
    for _ in range(n_attempts):
        if len(pairs) >= n_target: break
        i, j = int(rng.integers(n1)), int(rng.integers(n2))
        if (i, j) in seen or (n1 == n2 and i == j): continue
        seen.add((i, j))
        corr = float(np.corrcoef(wins1[i].flatten(), wins2[j].flatten())[0, 1])
        ok = True
        if corr_lo is not None and corr >= corr_lo: ok = False
        if corr_hi is not None and corr <= corr_hi: ok = False
        if ok: pairs.append((i, j))
    return pairs

rng_ctrl = np.random.default_rng(2024)

ctrl_same_x1, ctrl_same_x2, ctrl_same_subj = [], [], []
ctrl_diff_x1, ctrl_diff_x2, ctrl_diff_subj = [], [], []
viable_nm_ids = []

print('Building D1-curated pairs (same: corr<0.5, diff: corr>0.7)...')
for sid in nonmember_ids:
    idx  = np.where(y_d1_nm == sid)[0]
    wins = X_d1_nm[idx]

    # Same-person pairs: low correlation (different gait phases)
    same_pairs = [(i,j) for (i,j) in
                  sample_filtered_pairs(wins, wins, N_SAME_TARGET, corr_lo=SAME_CORR_THRESH, rng=rng_ctrl)
                  if i != j]
    if len(same_pairs) < MIN_PAIRS:
        log.info(f'  ID {sid}: {len(same_pairs)} low-corr pairs — excluded')
        continue
    viable_nm_ids.append(sid)

    # Diff-person pairs: high correlation (same gait phase, different person)
    other_idx  = np.where(y_d1_nm != sid)[0]
    wins_other = X_d1_nm[other_idx]
    diff_pairs = sample_filtered_pairs(wins, wins_other, N_DIFF_TARGET, corr_hi=DIFF_CORR_THRESH, rng=rng_ctrl)

    for i, j in same_pairs:
        ctrl_same_x1.append(wins[i]); ctrl_same_x2.append(wins[j]); ctrl_same_subj.append(sid)
    for i, j in diff_pairs:
        ctrl_diff_x1.append(wins[i]); ctrl_diff_x2.append(wins_other[j]); ctrl_diff_subj.append(sid)

excluded_ids = [s for s in nonmember_ids if s not in viable_nm_ids]
print(f'Viable: {len(viable_nm_ids)}/20  Excluded: {excluded_ids}')

ctrl_same_x1   = np.array(ctrl_same_x1);   ctrl_same_x2   = np.array(ctrl_same_x2)
ctrl_diff_x1   = np.array(ctrl_diff_x1);   ctrl_diff_x2   = np.array(ctrl_diff_x2)
ctrl_same_subj = np.array(ctrl_same_subj); ctrl_diff_subj = np.array(ctrl_diff_subj)

# Correlation diagnostics
n_d = min(400, len(ctrl_same_x1))
d_idx = rng_ctrl.choice(len(ctrl_same_x1), n_d, replace=False)
sc = [float(np.corrcoef(ctrl_same_x1[i].flatten(), ctrl_same_x2[i].flatten())[0,1]) for i in d_idx]
d_idx2 = rng_ctrl.choice(len(ctrl_diff_x1), min(n_d,len(ctrl_diff_x1)), replace=False)
dc = [float(np.corrcoef(ctrl_diff_x1[i].flatten(), ctrl_diff_x2[i].flatten())[0,1]) for i in d_idx2]
print(f'Controlled same corr: mean={np.mean(sc):.3f}  (D5 train ref: 0.234)')
print(f'Controlled diff corr: mean={np.mean(dc):.3f}  (D5 train ref: 0.762)')

# Normalize with D5 train stats and score
ctrl_same_x1_n = (ctrl_same_x1 - norm_mean) / norm_std
ctrl_same_x2_n = (ctrl_same_x2 - norm_mean) / norm_std
ctrl_diff_x1_n = (ctrl_diff_x1 - norm_mean) / norm_std
ctrl_diff_x2_n = (ctrl_diff_x2 - norm_mean) / norm_std

print('\nScoring D1-curated pairs...')
s_ctrl_same = score_pairs(ctrl_same_x1_n, ctrl_same_x2_n)
s_ctrl_diff = score_pairs(ctrl_diff_x1_n, ctrl_diff_x2_n)
print(f'Controlled same P(same): {s_ctrl_same.mean():.3f}  diff P(same): {s_ctrl_diff.mean():.3f}')
log.info(f'Controlled: same P(same)={s_ctrl_same.mean():.3f}  diff P(same)={s_ctrl_diff.mean():.3f}')
log.info(f'Viable non-member subjects: {len(viable_nm_ids)}/20  excluded: {excluded_ids}')

In [ ]:
# ── Per-subject delta for controlled non-members ──
ctrl_nm_deltas = {}
for sid in viable_nm_ids:
    pos = s_ctrl_same[ctrl_same_subj == sid]
    neg = s_ctrl_diff[ctrl_diff_subj == sid]
    if len(pos) > 0 and len(neg) > 0:
        ctrl_nm_deltas[sid] = {
            'delta':    float(pos.mean() - neg.mean()),
            'pos_mean': float(pos.mean()),
            'neg_mean': float(neg.mean()),
            'n_pos': len(pos), 'n_neg': len(neg),
        }

ctrl_nm_arr = np.array([v['delta'] for v in ctrl_nm_deltas.values()])

print(f'Members (D5 train):       mean={m_arr.mean():.4f}  std={m_arr.std():.4f}  >0: {(m_arr>0).sum()}/{len(m_arr)}')
print(f'Non-member D5 test:       mean={nm_arr.mean():.4f}  std={nm_arr.std():.4f}  >0: {(nm_arr>0).sum()}/{len(nm_arr)}')
print(f'Non-member D1 ctrl:       mean={ctrl_nm_arr.mean():.4f}  std={ctrl_nm_arr.std():.4f}  >0: {(ctrl_nm_arr>0).sum()}/{len(ctrl_nm_arr)}')

# ── Controlled AUC ──
ctrl_scores = np.concatenate([m_arr, ctrl_nm_arr])
ctrl_labels = np.concatenate([np.ones(len(m_arr)), np.zeros(len(ctrl_nm_arr))])
ctrl_fpr, ctrl_tpr, _ = roc_curve(ctrl_labels, ctrl_scores)
ctrl_auc = sk_auc(ctrl_fpr, ctrl_tpr)
ctrl_gap  = float(m_arr.mean() - ctrl_nm_arr.mean())

print(f'\nOriginal  MIA AUC (D5, session confound):    {mia_auc:.4f}')
print(f'Controlled MIA AUC (D1 session-matched):     {ctrl_auc:.4f}')
print(f'Controlled delta gap (member - non-member):  {ctrl_gap:.4f}')
print(f'\n→ Session confound was HELPING non-members (cross-session D5 test gave delta=0.503);')
print(f'  with session-matched D1 pairs, non-member delta collapses to {ctrl_nm_arr.mean():.3f}.')
print(f'  Membership signal survives: controlled AUC={ctrl_auc:.4f} > 0.5.')

log.info(f'Controlled MIA AUC: {ctrl_auc:.4f}  gap: {ctrl_gap:.4f}')

# ── Figure: delta distribution comparison + dual ROC ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: delta distributions
ax = axes[0]
bins = np.linspace(-0.6, 1.05, 30)
ax.hist(nm_arr,      bins=bins, alpha=0.55, color='#e74c3c',
        label=f'Non-member D5 test   (n={len(nm_arr)})  mean={nm_arr.mean():.3f}', density=True)
ax.hist(ctrl_nm_arr, bins=bins, alpha=0.55, color='#e67e22',
        label=f'Non-member D1 ctrl   (n={len(ctrl_nm_arr)})  mean={ctrl_nm_arr.mean():.3f}', density=True)
ax.hist(m_arr,       bins=bins, alpha=0.55, color='#3498db',
        label=f'Members D5 train     (n={len(m_arr)})  mean={m_arr.mean():.3f}', density=True)
ax.axvline(m_arr.mean(),       color='#3498db', linestyle='--', linewidth=1.5)
ax.axvline(nm_arr.mean(),      color='#e74c3c', linestyle='--', linewidth=1.5)
ax.axvline(ctrl_nm_arr.mean(), color='#e67e22', linestyle='--', linewidth=1.5)
ax.set_xlabel('Delta score'); ax.set_ylabel('Density')
ax.set_title('Delta distributions: members vs non-members (two evaluation protocols)')
ax.legend(fontsize=8)

# Right: ROC comparison
ax = axes[1]
ax.plot(fpr,      tpr,      color='#e74c3c', linewidth=2,
        label=f'D5 pairs — session confound  (AUC={mia_auc:.3f})')
ax.plot(ctrl_fpr, ctrl_tpr, color='#e67e22', linewidth=2, linestyle='--',
        label=f'D1-curated — session matched (AUC={ctrl_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.4, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('MIA ROC — D5 pairs vs session-controlled')
ax.legend(fontsize=9); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
out_path = RESULTS_DIR / '04_controlled_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
log.info(f'Saved: {out_path}')

In [ ]:
# ── Update LaTeX macros — add controlled evaluation metrics ──
write_latex_metrics('nb04', {
    # Original D5-based (keep complete)
    'miaMemberAttributed':          n_m_attr,
    'miaMemberTotal':               len(member_ids),
    'miaNonMemberAttributed':       n_nm_attr,
    'miaD5SamePairCorr':            f'{corr_tr_same.mean():.3f}',
    'miaD5DiffPairCorr':            f'{corr_tr_diff.mean():.3f}',
    'miaMemberSameMean':            f'{tr_same_mean:.3f}',
    'miaMemberDiffMean':            f'{tr_diff_mean:.3f}',
    'miaMemberAggregateDelta':      f'{tr_same_mean-tr_diff_mean:.3f}',
    'miaNonMemberSameMean':         f'{te_same_mean:.3f}',
    'miaNonMemberDiffMean':         f'{te_diff_mean:.3f}',
    'miaNonMemberAggregateDelta':   f'{te_same_mean-te_diff_mean:.3f}',
    'miaMemberDeltaMean':           f'{m_arr.mean():.4f}',
    'miaMemberDeltaStd':            f'{m_arr.std():.4f}',
    'miaNonMemberDeltaMean':        f'{nm_arr.mean():.4f}',
    'miaNonMemberDeltaStd':         f'{nm_arr.std():.4f}',
    'miaDeltaGap':                  f'{delta_gap:.4f}',
    'miaAUC':                       f'{mia_auc:.4f}',
    'miaOptimalThreshold':          f'{opt_thr:.4f}',
    'miaOptimalTPR':                f'{opt_tpr:.3f}',
    'miaOptimalFPR':                f'{opt_fpr:.3f}',
    'miaTPRatFPR10':                f'{tpr_at_10:.3f}',
    'miaTPRatFPR20':                f'{tpr_at_20:.3f}',
    # Controlled evaluation (new)
    'miaControlledAUC':             f'{ctrl_auc:.4f}',
    'miaControlledNonMemberDelta':  f'{ctrl_nm_arr.mean():.4f}',
    'miaControlledDeltaGap':        f'{ctrl_gap:.4f}',
    'miaControlledViable':          len(ctrl_nm_deltas),
    'miaControlledExcluded':        len(excluded_ids),
}, output_dir='../latex/generated', log=log)
print('LaTeX macros updated.')

## Results

The delta score separates members from non-members with AUC 0.8435 using only the authentication output.

| Group | Delta mean | Delta std | n subjects |
|---|---|---|---|
| Members | 0.763 | 0.112 | 84 (of 98) |
| Non-members | 0.503 | 0.201 | 20 |
| Gap | 0.259 | - |: |

14 of the 98 training subjects are not attributed because their windows do not appear in the D5 train set with enough pair coverage.

An AUC of 0.843 is a solid result given the constraints: no model internals, no gradients, no per-class logits: just P(same) from the authentication interface. For comparison, a random classifier gives 0.50 and the Milani WIFS 2024 paper achieves ~0.977 using the CNN identification logit directly, which requires internal model access.

At the Youden-optimal threshold (delta = 0.664), the attack correctly identifies 70 of 84 members (TPR = 0.833) with 4 false positives (FPR = 0.200). The stricter operating point is more revealing: at FPR = 0.10, the TPR drops to 0.226, meaning fewer than 1 in 4 members is caught before 2 non-members are wrongly flagged. This is the weakness that LiRA addresses in NB05 by calibrating each subject's score against shadow-model expectations rather than applying a global threshold.

The session-controlled experiment confirms the signal is genuine. When non-members are evaluated on D1-curated pairs (removing the session difference), the membership gap grows to 0.755 and the controlled AUC rises to 0.994. Cross-session difficulty was slightly inflating the non-member delta in the main evaluation, but the underlying memorisation signal is real.

Four non-members (IDs 1, 7, 10, 14) produce delta scores between 0.74 and 0.90 despite never being in training. Their gait patterns are close enough to the training distribution that the authentication model scores them as members regardless. These represent a hard ceiling for any delta-based attack and are examined more closely in NB05.